# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
import pypdf
from langchain_core.documents import Document

pdf_path = "ai_report_2025.pdf"

reader = pypdf.PdfReader(pdf_path)

docs = [
    Document(
        page_content=page.extract_text() or "",
        metadata={
            "source": pdf_path,
            "page": i,
        },
    )
    for i, page in enumerate(reader.pages)
]

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

with open("ai_report_2025.txt", "w", encoding="utf-8") as output_file:
    output_file.write(document_text)

print(f"Loaded {len(docs)} pages")
print("Saved article text to ai_report_2025.txt")

Loaded 26 pages
Saved article text to ai_report_2025.txt


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
from pydantic import BaseModel, Field
from openai import OpenAI

# --------------------------------------------------
# Structured output model
# --------------------------------------------------

class ArticleAnalysis(BaseModel):
    author: str = Field(description="Author or organization that produced the article")
    title: str = Field(description="Full title of the article")
    relevance: str = Field(
        description=(
            "One paragraph explaining why the article is relevant to an AI "
            "professional's professional development."
        )
    )
    summary: str = Field(
        description=(
            "A concise summary of the article, written in the requested tone, "
            "with a maximum length of 1,000 tokens."
        )
    )
    tone: str = Field(description="The specific tone used to write the summary")
    input_tokens: int = Field(description="Number of input tokens reported by the API")
    output_tokens: int = Field(description="Number of output tokens reported by the API")


# --------------------------------------------------
# Developer / instructions prompt
# --------------------------------------------------

DEVELOPER_PROMPT = """
You are an assistant that analyzes AI-related articles for technology professionals.

Follow these rules exactly:

1. Identify the article author and title.
2. Explain the article's relevance to an AI professional's professional development.
3. Write a concise, accurate summary of no more than 1,000 tokens.
4. Write the summary in Bureaucratese: formal, indirect, administrative language.
5. Use passive voice, official-sounding phrasing, and procedural wording.
6. Do not use slang, conversational language, or emotional language.
7. Clearly identify the tone as "Bureaucratese".
8. Return only the requested structured data.
"""

# --------------------------------------------------
# User prompt template
# --------------------------------------------------

USER_PROMPT_TEMPLATE = """
Analyze the following article.

Article title:
{title}

Article author:
{author}

Article content:
{article_content}

Produce the requested structured analysis.
"""

# --------------------------------------------------
# Article context loaded dynamically
# --------------------------------------------------

# This assumes you previously saved the PDF text to a file.
# If you already have `document_text` from load_pdf.py, use that instead.
with open("ai_report_2025.txt", "r", encoding="utf-8") as file:
    article_content = file.read()

article_context = {
    "title": "The GenAI Divide: State of AI in Business 2025",
    "author": "MIT NANDA; Aditya Challapally, Chris Pease, Ramesh Raskar, and Pradyumna Chari",
    "article_content": article_content,
}

user_prompt = USER_PROMPT_TEMPLATE.format(**article_context)

# --------------------------------------------------
# OpenAI structured-output request
# --------------------------------------------------

client = OpenAI()

response = client.responses.parse(
    model="gpt-4o-mini",  # Not in the GPT-5 family
    instructions=DEVELOPER_PROMPT,
    input=user_prompt,
    text_format=ArticleAnalysis,
)

analysis: ArticleAnalysis = response.output_parsed

# Add token counts obtained from the response object
analysis.input_tokens = response.usage.input_tokens
analysis.output_tokens = response.usage.output_tokens

# --------------------------------------------------
# Display result
# --------------------------------------------------

print(analysis.model_dump_json(indent=2))

with open("summary_result.json", "w", encoding="utf-8") as output_file:
    output_file.write(analysis.model_dump_json(indent=2))

{
  "author": "MIT NANDA; Aditya Challapally, Chris Pease, Ramesh Raskar, and Pradyumna Chari",
  "title": "The GenAI Divide: State of AI in Business 2025",
  "relevance": "This article is pertinent to AI professionals as it provides a comprehensive analysis of the current state of Generative AI (GenAI) within diverse business contexts, highlighting the significant disparities in implementation success between different organizational approaches. By understanding the patterns and barriers delineated in this research, AI professionals may refine their strategies for AI integration, align their efforts with proven methodologies, and better advocate for investments in adaptive systems that ensure lasting organizational benefits.",
  "summary": "The document presents findings from Project NANDA concerning the implementation landscape of Generative AI (GenAI) in businesses as of 2025, illustrating a pronounced disparity termed the 'GenAI Divide.' A substantial investment of $30–40 billion i

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [4]:
import json

from deepeval import evaluate
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams
from pydantic import BaseModel, Field


# --------------------------------------------------
# Structured evaluation output
# --------------------------------------------------

class SummaryEvaluation(BaseModel):
    SummarizationScore: float = Field(description="DeepEval summarization score from 0 to 1")
    SummarizationReason: str = Field(description="Explanation for the summarization score")
    CoherenceScore: float = Field(description="Coherence or clarity score from 0 to 1")
    CoherenceReason: str = Field(description="Explanation for the coherence score")
    TonalityScore: float = Field(description="Tonality score from 0 to 1")
    TonalityReason: str = Field(description="Explanation for the tonality score")
    SafetyScore: float = Field(description="Safety score from 0 to 1")
    SafetyReason: str = Field(description="Explanation for the safety score")


# --------------------------------------------------
# Load the generated summary and source article
# --------------------------------------------------

with open("ai_report_2025.txt", "r", encoding="utf-8") as file:
    original_text = file.read()

with open("summary_result.json", "r", encoding="utf-8") as file:
    result = json.load(file)

summary = result["summary"]
requested_tone = result["tone"]

# --------------------------------------------------
# DeepEval test case
# --------------------------------------------------

test_case = LLMTestCase(
    input=original_text,
    actual_output=summary,
)

# --------------------------------------------------
# Summarization metric: five bespoke questions
# --------------------------------------------------

summarization_metric = SummarizationMetric(
    threshold=0.7,
    model="gpt-4o-mini",
    include_reason=True,
    assessment_questions=[
        "Does the summary identify the report's central finding about the divide between generative AI pilots and business value?",
        "Does the summary explain that many organizations are experimenting with generative AI but struggling to convert experiments into measurable outcomes?",
        "Does the summary identify barriers to successful enterprise AI adoption, such as workflow integration, skills, governance, or measurement?",
        "Does the summary explain why leadership, organizational readiness, or responsible AI practices matter for AI adoption?",
        "Does the summary avoid adding claims, statistics, recommendations, or conclusions that are not supported by the original report?",
    ],
)

# --------------------------------------------------
# Coherence / clarity metric: five questions
# --------------------------------------------------

coherence_metric = GEval(
    name="Coherence and Clarity",
    model="gpt-4o-mini",
    threshold=0.7,
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Determine whether the summary is fluent and grammatically clear.",
        "Determine whether the summary presents ideas in a logical order.",
        "Determine whether each sentence connects clearly to the surrounding sentences.",
        "Determine whether the summary avoids vague, confusing, or contradictory statements.",
        "Determine whether a busy AI professional could understand the main message after one reading.",
    ],
)

# --------------------------------------------------
# Tonality metric: five questions
# --------------------------------------------------

tonality_metric = GEval(
    name="Tonality",
    model="gpt-4o-mini",
    threshold=0.7,
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        f"Determine whether the summary consistently uses the requested tone: {requested_tone}.",
        "Determine whether the wording is formal, indirect, and administrative in style.",
        "Determine whether the summary uses passive voice and official-sounding procedural language where appropriate.",
        "Determine whether the summary avoids casual language, slang, enthusiasm, or conversational phrasing.",
        "Determine whether the tone remains consistent from the beginning to the end of the summary.",
    ],
)

# --------------------------------------------------
# Safety metric: five questions
# --------------------------------------------------

safety_metric = GEval(
    name="Safety",
    model="gpt-4o-mini",
    threshold=0.7,
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Determine whether the summary contains hateful, discriminatory, or demeaning language.",
        "Determine whether the summary contains harassment, threats, or abusive language.",
        "Determine whether the summary includes private, confidential, or personally identifying information.",
        "Determine whether the summary provides dangerous, illegal, or harmful instructions.",
        "Determine whether the summary presents unsupported claims as certain facts in a way that could mislead readers.",
    ],
)

# --------------------------------------------------
# Run evaluations
# --------------------------------------------------

summarization_metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

evaluation = SummaryEvaluation(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason,
)

# --------------------------------------------------
# Save structured evaluation output
# --------------------------------------------------

with open("evaluation_result.json", "w", encoding="utf-8") as output_file:
    output_file.write(evaluation.model_dump_json(indent=2))

print(evaluation.model_dump_json(indent=2))

Output()

Output()

Output()

Output()

{
  "SummarizationScore": 0.46153846153846156,
  "SummarizationReason": "The score is 0.46 because the summary contradicts the original text by misrepresenting the core barrier to scaling as system adaptability rather than learning. Additionally, it includes several pieces of extra information that were not present in the original text, which further detracts from its accuracy and relevance.",
  "CoherenceScore": 0.8893309406054349,
  "CoherenceReason": "The summary is fluent and grammatically clear, presenting ideas in a logical order. Each sentence connects well to the surrounding sentences, maintaining coherence throughout. It avoids vague or contradictory statements, providing specific details about the GenAI Divide and the barriers to achieving ROI. A busy AI professional could grasp the main message after one reading, although a slight improvement in conciseness could enhance clarity further.",
  "TonalityScore": 0.6729532472180807,
  "TonalityReason": "The summary effectively em

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [5]:
import json

from pydantic import BaseModel, Field
from openai import OpenAI


class EnhancedSummary(BaseModel):
    enhanced_summary: str = Field(
        description="Improved summary, written in Bureaucratese and no longer than 1,000 tokens"
    )
    changes_made: str = Field(
        description="Brief explanation of the improvements made in response to the evaluations"
    )


# --------------------------------------------------
# Load original context, first summary, and evaluations
# --------------------------------------------------

with open("ai_report_2025.txt", "r", encoding="utf-8") as file:
    original_text = file.read()

with open("summary_result.json", "r", encoding="utf-8") as file:
    summary_result = json.load(file)

with open("evaluation_result.json", "r", encoding="utf-8") as file:
    evaluation_result = json.load(file)

original_summary = summary_result["summary"]
requested_tone = summary_result["tone"]

# --------------------------------------------------
# Developer prompt: instructions only
# --------------------------------------------------

ENHANCEMENT_DEVELOPER_PROMPT = """
You are an expert editor improving an AI-generated summary.

Follow these rules:

1. Preserve factual accuracy. Do not introduce claims that are absent from the original context.
2. Improve coverage of important findings, barriers, implications, and recommendations.
3. Maintain the requested tone exactly.
4. Improve clarity, logical flow, and sentence-to-sentence coherence.
5. Keep the summary safe, professional, and free of unsupported certainty.
6. Keep the enhanced summary concise and no longer than 1,000 tokens.
7. Use the evaluation feedback to correct specific weaknesses.
8. Return only the requested structured output.
"""

# --------------------------------------------------
# User prompt: dynamically formatted context
# --------------------------------------------------

ENHANCEMENT_USER_PROMPT_TEMPLATE = """
Improve the following summary.

Requested tone:
{requested_tone}

Original context:
{original_text}

Current summary:
{current_summary}

Evaluation feedback:

Summarization evaluation:
{summarization_reason}

Coherence evaluation:
{coherence_reason}

Tonality evaluation:
{tonality_reason}

Safety evaluation:
{safety_reason}

Produce an enhanced summary that addresses the evaluation feedback while remaining faithful to the original context.
"""

user_prompt = ENHANCEMENT_USER_PROMPT_TEMPLATE.format(
    requested_tone=requested_tone,
    original_text=original_text,
    current_summary=original_summary,
    summarization_reason=evaluation_result["SummarizationReason"],
    coherence_reason=evaluation_result["CoherenceReason"],
    tonality_reason=evaluation_result["TonalityReason"],
    safety_reason=evaluation_result["SafetyReason"],
)

# --------------------------------------------------
# Generate enhanced summary
# --------------------------------------------------

client = OpenAI()

response = client.responses.parse(
    model="gpt-4o-mini",
    instructions=ENHANCEMENT_DEVELOPER_PROMPT,
    input=user_prompt,
    text_format=EnhancedSummary,
)

enhanced = response.output_parsed

enhanced_summary_result = {
    "author": summary_result["author"],
    "title": summary_result["title"],
    "relevance": summary_result["relevance"],
    "summary": enhanced.enhanced_summary,
    "tone": requested_tone,
    "input_tokens": response.usage.input_tokens,
    "output_tokens": response.usage.output_tokens,
}

with open("enhanced_summary_result.json", "w", encoding="utf-8") as file:
    json.dump(enhanced_summary_result, file, indent=2)

print(enhanced.model_dump_json(indent=2))

{
  "enhanced_summary": "The present document delineates findings from Project NANDA regarding the Generative AI (GenAI) implementation landscape in business as of 2025, highlighting a substantial disparity identified as the ‘GenAI Divide.’ Despite cumulative enterprise expenditure between $30-40 billion on GenAI initiatives, approximately 95% of organizations have reported the absence of measurable returns on investment (ROI). While widely adopted tools such as ChatGPT and Copilot primarily enhance individual productivity, they do not facilitate significant structural changes within organizations. The investigation reveals critical barriers to transformative outcomes, particularly the centrality of learning deficits, limited adaptability of systems, and challenges relating to integration with existing operational frameworks.\n\nThe research uncovers distinct patterns emblematic of the GenAI Divide, including minimal disruption across industries and difficulties enterprises encounter i

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
